# JobShield – 03 Baseline Model (Week 3)
**Group 5 · AML-3403 AI and ML Capstone · Week 3 lead: Diksha Sharma** · https://github.com/Krushal16/JobShield

1. Train the baselines – TF-IDF + Logistic Regression and TF-IDF + Naive Bayes
2. Confusion matrix, precision-recall curves, threshold trade-off
3. TF-IDF tuning (5-fold CV on the training set)
4. Words the model relies on
5. Error analysis – missed scams and false alarms
6. Save the model and score single postings with `predict_posting()`
7. Summary

**Needs** `data/processed/train.csv` and `val.csv` from notebook 02. The test set is not used until Week 7.

In [2]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.data.loader import TARGET
from src.data.split import load_splits
from src.training.train_baseline import train_and_evaluate, save_model, top_features
from src.training.tune_tfidf import tune_tfidf
from src.evaluation.metrics import plot_confusion, plot_pr_curve
from src.evaluation.error_analysis import error_table, threshold_table, missed_scam_profile, plot_top_features
from src.inference.predict import predict_posting, load_model
from src.utils.config import load_config, set_seed
CFG = load_config(); set_seed(CFG["seed"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
train_df, val_df, _ = load_splits()
print(f"train {len(train_df):,} ({train_df[TARGET].sum()} fake) | val {len(val_df):,} ({val_df[TARGET].sum()} fake)")
summary = {}

ModuleNotFoundError: No module named 'src.evaluation.error_analysis'

## 1. Train the baselines

In [ ]:
out = train_and_evaluate(train_df, val_df, CFG, log=True)
pd.DataFrame(out["results"])[["model", "notes", "precision", "recall", "f1", "pr_auc", "tp", "fn", "fp"]].round(3)

## 2. Confusion matrix, precision-recall curves and threshold trade-off

In [ ]:
LR, NB = "TF-IDF + Logistic Regression", "TF-IDF + Naive Bayes"
y_val = val_df[TARGET].values
p_lr, p_nb, thr_lr = out["probs"][LR], out["probs"][NB], out["thresholds"][LR]
lr_best = [r for r in out["results"] if r["model"] == LR and "F1-best" in r["notes"]][0]
plot_confusion(lr_best, save_as="09_baseline_confusion.png"); plt.show()
fig, ax = plt.subplots(figsize=(5, 4))
plot_pr_curve(y_val, p_lr, label="Logistic Regression", ax=ax)
plot_pr_curve(y_val, p_nb, label="Naive Bayes", ax=ax, save_as="10_baseline_pr_curves.png"); plt.show()
threshold_table(y_val, p_lr)

## 3. TF-IDF tuning (5-fold CV on the training set only – a few minutes)

In [ ]:
tuning = tune_tfidf(train_df, CFG)
tuning.head(6)

## 4. Words the model relies on

In [ ]:
pipe_lr = out["models"][LR]
top = top_features(pipe_lr, n=20)
plot_top_features(top, n=15, save_as="11_baseline_top_words.png"); plt.show()

## 5. Error analysis

In [ ]:
print("Missed scams:", int(((y_val == 1) & (p_lr < thr_lr)).sum()), "| False alarms:", int(((y_val == 0) & (p_lr >= thr_lr)).sum()))
print(missed_scam_profile(val_df, p_lr, threshold=thr_lr))
error_table(val_df, p_lr, threshold=thr_lr, kind="fn")

## 6. Save the model and score single postings

In [ ]:
bundle = load_model(save_model(pipe_lr, thr_lr, cfg=CFG))
r = predict_posting(title="Data Entry Assistant – Work From Home",
                    description="Earn $500 weekly from home. No experience needed. Contact us on WhatsApp to start immediately.",
                    has_company_logo=0, model_bundle=bundle)
print(r)

## 7. Summary

In [ ]:
for r in out["results"]:
    print(f"{r['model']:30s} {r['notes']:28s} P={r['precision']:.3f} R={r['recall']:.3f} F1={r['f1']:.3f} PR-AUC={r['pr_auc']:.3f}")